# Iniciando o Spark



In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
from pyspark.sql import SparkSession
import os
import pytz
from datetime import datetime

spark = SparkSession.builder.appName("tabelas").getOrCreate()

In [3]:
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [4]:
spark

# Instalando bibliotecas

In [5]:

import os
import sys
import pytz
import numpy as np
import datetime
from pyspark.sql import SparkSession
from pyspark.sql import SQLContext
from pyspark.sql.functions import udf,split, lpad, concat_ws,to_timestamp,col,coalesce
from datetime import datetime
from datetime import timedelta
from datetime import date
from dateutil.relativedelta import relativedelta
from pyspark.sql.types import *
from pyspark.sql.functions import count, avg, to_date
from pyspark.sql import functions as F

#Configuração do pipeline( contem configuração do spark e os paths para serem alterados)


In [6]:
# caminhos dos arquivos

PATH_TABELA_BUREAU ="/content/gdrive/MyDrive/Raw Hackathon PoD 2025/base_score_bureau_movel_full"
PATH_TABELA_CADASTRO = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/base_dados_cadastrais"
PATH_BASE_TELCO       = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/base_telco"
PATH_BASE_RECARGA     = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/bases_recarga/BI_FP_ASS_RECARGA_CMV_NOVA"
PATH_BASE_BOOK_ATRASO = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/book_atraso/dados_faturamento"
PATH_BASE_BOOK_PAGAMENTO ="/content/gdrive/MyDrive/Raw Hackathon PoD 2025/book_pagamento/dados_pagamento"

BASE_PATH_RECARGA = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/bases_recarga"

PATH_DIMENSOES_RECARGA = {
    "CANAL_AQUISICAO": f"{BASE_PATH_RECARGA}/BI_DIM_CANAL_AQUISICAO_CREDITO.csv",
    "FORMA_PAGAMENTO": f"{BASE_PATH_RECARGA}/BI_DIM_FORMA_PAGAMENTO.csv",
    "INSTITUICAO": f"{BASE_PATH_RECARGA}/BI_DIM_INSTITUICAO.csv",
    "PLATAFORMA": f"{BASE_PATH_RECARGA}/BI_DIM_PLATAFORMA.csv",
    "PROMOCAO": f"{BASE_PATH_RECARGA}/BI_DIM_PROMOCAO_CREDITO.csv",
    "STATUS_PLATAFORMA": f"{BASE_PATH_RECARGA}/BI_DIM_STATUS_PLATAFORMA.csv",
    "TECNOLOGIA": f"{BASE_PATH_RECARGA}/BI_DIM_TECNOLOGIA.csv",
    "TIPO_CREDITO": f"{BASE_PATH_RECARGA}/BI_DIM_TIPO_CREDITO.csv",
    "TIPO_INSERCAO": f"{BASE_PATH_RECARGA}/BI_DIM_TIPO_INSERCAO.csv",
    "TIPO_RECARGA": f"{BASE_PATH_RECARGA}/BI_DIM_TIPO_RECARGA.csv",
}


PATH_DIMENSOES_BOOK_ATRASO = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/book_atraso/BI_DIM_TIPO_FATURAMENTO.csv"

In [7]:
# Csvs da tabela recarga

BASE_PATH_RECARGA = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/bases_recarga"

DIMENSOES_RECARGA = {
    "CANAL_AQUISICAO_CREDITO": "BI_DIM_CANAL_AQUISICAO_CREDITO.csv",
    "FORMA_PAGAMENTO": "BI_DIM_FORMA_PAGAMENTO.csv",
    "INSTITUICAO": "BI_DIM_INSTITUICAO.csv",
    "PLANO_PRECO": "BI_DIM_PLANO_PRECO.csv",
    "PLATAFORMA": "BI_DIM_PLATAFORMA.csv",
    "PROMOCAO_CREDITO": "BI_DIM_PROMOCAO_CREDITO.csv",
    "STATUS_PLATAFORMA": "BI_DIM_STATUS_PLATAFORMA.csv",
    "TECNOLOGIA": "BI_DIM_TECNOLOGIA.csv",
    "TIPO_CREDITO": "BI_DIM_TIPO_CREDITO.csv",
    "TIPO_INSERCAO": "BI_DIM_TIPO_INSERCAO.csv",
    "TIPO_RECARGA": "BI_DIM_TIPO_RECARGA.csv",
}


dfs_dim_recarga = {}

for nome_dim, arquivo in DIMENSOES_RECARGA.items():
    path = f"{BASE_PATH_RECARGA}/{arquivo}"

    dfs_dim_recarga[nome_dim] = (
        spark.read
        .option("header", True)
        .option("sep", ",")
        .option("inferSchema", True)
        .csv(path)
    )

df_CANAL_AQUISICAO_CREDITO = dfs_dim_recarga["CANAL_AQUISICAO_CREDITO"]
df_FORMA_PAGAMENTO        = dfs_dim_recarga["FORMA_PAGAMENTO"]
df_INSTITUICAO            = dfs_dim_recarga["INSTITUICAO"]
df_PLANO_PRECO            = dfs_dim_recarga["PLANO_PRECO"]
df_PLATAFORMA             = dfs_dim_recarga["PLATAFORMA"]
df_PROMOCAO_CREDITO       = dfs_dim_recarga["PROMOCAO_CREDITO"]
df_STATUS_PLATAFORMA      = dfs_dim_recarga["STATUS_PLATAFORMA"]
df_TECNOLOGIA              = dfs_dim_recarga["TECNOLOGIA"]
df_TIPO_CREDITO           = dfs_dim_recarga["TIPO_CREDITO"]
df_TIPO_INSERCAO          = dfs_dim_recarga["TIPO_INSERCAO"]
df_TIPO_RECARGA           = dfs_dim_recarga["TIPO_RECARGA"]

In [8]:
# cadastro
df_cadastro = spark.read.parquet(PATH_TABELA_CADASTRO)
df_cadastro.createOrReplaceTempView("df_cadastro")

# bureau
df_bureau = spark.read.parquet(PATH_TABELA_BUREAU)
df_bureau.createOrReplaceTempView("df_bureau")

# telco
df_base_telco = spark.read.parquet(PATH_BASE_TELCO)
df_base_telco.createOrReplaceTempView("df_base_telco")

# recarga
df_recarga = spark.read.parquet(PATH_BASE_RECARGA)
df_recarga.createOrReplaceTempView("df_recarga")

# book atraso
df_book_atraso = spark.read.parquet(PATH_BASE_BOOK_ATRASO)
df_book_atraso.createOrReplaceTempView("df_book_atraso")

# book pagamento
df_book_pagamento = spark.read.parquet(PATH_BASE_BOOK_PAGAMENTO)
df_book_pagamento.createOrReplaceTempView("df_book_pagamento")


# Analise Inicial dos dados da tabela recarga

Foi executada em ambiente pyspark devido ao volume de dados


Análise da Estrutura e Qualidade dos Dados

Principais ajustes realizados

Identificadores (NUM_CPF, DW_NUM_NTC, DW_NUM_CLIENTE)
Mantidos como string, evitando perda de informação e preservando zeros à esquerda.

Campos temporais (DAT_INSERCAO_CREDITO, HOR_INSERCAO_CREDITO)
Mantidos como string nesta etapa, com previsão de conversão posterior para date e timestamp na camada de enriquecimento temporal.

A coluna DAT_INSERCAO_CREDITO foi convertida do tipo string para date, com o objetivo de padronizar o campo temporal e permitir sua utilização na criação da variável SAFRA.

Códigos de dimensão (COD_CANAL_AQUISICAO, COD_PROMOCAO, DW_PLANO_TARIFACAO, DW_TIPO_RECARGA, DW_TIPO_INSERCAO, DW_FORMA_PAGAMENTO, DW_INSTITUICAO)
Convertidos para int, permitindo:

Melhor performance em joins

Redução de armazenamento

Padronização para modelo estrela

Valores financeiros (VAL_CREDITO_INSERIDO, VAL_BONUS, VAL_REAL)
Convertidos para float, possibilitando cálculos, agregações e análises financeiras.

Flags e indicadores (FLAG_SOS)
Convertidos para int (0/1), facilitando filtros e métricas operacionais.

Campos descritivos (COD_GRUPO_CARTAO, DSC_GRUPO_CARTAO_WPP)
Mantidos como string por representarem descrições categóricas.

In [ ]:
# recarga
df_recarga = spark.read.parquet(PATH_BASE_RECARGA)
df_recarga.createOrReplaceTempView("df_recarga")


In [ ]:
# Mudando o datatype
df_base_recarga = spark.sql("""

    SELECT
        CAST(NUM_CPF AS STRING) AS NUM_CPF,
        CAST(DW_NUM_NTC AS STRING) AS DW_NUM_NTC,
        CAST(DW_NUM_CLIENTE AS STRING) AS DW_NUM_CLIENTE,

        -- DATA de recarga sem hora (truncada para mês)
        TRUNC(
            TO_TIMESTAMP(DAT_INSERCAO_CREDITO, 'ddMMMyyyy:HH:mm:ss'),
            'MM'
        ) AS DAT_INSERCAO_CREDITO,

        CAST(HOR_INSERCAO_CREDITO AS STRING) AS HOR_INSERCAO_CREDITO,
        CAST(COD_TECNOLOGIA_DW AS STRING) AS COD_TECNOLOGIA_DW,
        CAST(COD_CANAL_AQUISICAO AS INT) AS COD_CANAL_AQUISICAO,
        CAST(COD_TIPO_CREDITO AS STRING) AS COD_TIPO_CREDITO,
        CAST(COD_PROMOCAO AS INT) AS COD_PROMOCAO,
        CAST(VAL_CREDITO_INSERIDO AS FLOAT) AS VAL_CREDITO_INSERIDO,
        CAST(VAL_BONUS AS FLOAT) AS VAL_BONUS,
        CAST(VAL_REAL AS FLOAT) AS VAL_REAL,
        CAST(COD_PLATAFORMA_ATU AS STRING) AS COD_PLATAFORMA_ATU,
        CAST(COD_STATUS_PLATAFORMA AS STRING) AS COD_STATUS_PLATAFORMA,
        CAST(IND_METODO_PAGAMENTO AS STRING) AS IND_METODO_PAGAMENTO,
        CAST(DW_PLANO_TARIFACAO AS INT) AS DW_PLANO_TARIFACAO,
        CAST(DW_TIPO_RECARGA AS INT) AS DW_TIPO_RECARGA,
        CAST(DW_TIPO_INSERCAO AS INT) AS DW_TIPO_INSERCAO,
        CAST(DW_FORMA_PAGAMENTO AS INT) AS DW_FORMA_PAGAMENTO,
        CAST(DW_INSTITUICAO AS INT) AS DW_INSTITUICAO,
        CAST(COD_GRUPO_CARTAO AS STRING) AS COD_GRUPO_CARTAO,
        CAST(DSC_GRUPO_CARTAO_WPP AS STRING) AS DSC_GRUPO_CARTAO_WPP,
        CAST(FLAG_SOS AS INT) AS FLAG_SOS,
        CAST(VALOR_SOS AS INT) AS VALOR_SOS

    FROM df_base_recarga
""")



In [ ]:
# criando coluna SAFRA
df_base_recarga.createOrReplaceTempView("df_base_recarga")

df_base_recarga = spark.sql("""
SELECT
    f.*,
    trunc(
        to_timestamp(f.DAT_INSERCAO_CREDITO, 'ddMMMyyyy:HH:mm:ss'),
        'MM'
    ) AS SAFRA
FROM df_base_recarga f
""")

for c in df_base_recarga.columns:
    df_base_recarga = df_base_recarga.withColumnRenamed(
        c, f"B_RECARGA_{c}"
    )


In [ ]:
df_base_recarga.printSchema()

root
 |-- B_RECARGA_NUM_CPF: string (nullable = true)
 |-- B_RECARGA_DW_NUM_NTC: string (nullable = true)
 |-- B_RECARGA_DW_NUM_CLIENTE: string (nullable = true)
 |-- B_RECARGA_DAT_INSERCAO_CREDITO: date (nullable = true)
 |-- B_RECARGA_HOR_INSERCAO_CREDITO: string (nullable = true)
 |-- B_RECARGA_COD_TECNOLOGIA_DW: string (nullable = true)
 |-- B_RECARGA_COD_CANAL_AQUISICAO: integer (nullable = true)
 |-- B_RECARGA_COD_TIPO_CREDITO: string (nullable = true)
 |-- B_RECARGA_COD_PROMOCAO: integer (nullable = true)
 |-- B_RECARGA_VAL_CREDITO_INSERIDO: float (nullable = true)
 |-- B_RECARGA_VAL_BONUS: float (nullable = true)
 |-- B_RECARGA_VAL_REAL: float (nullable = true)
 |-- B_RECARGA_COD_PLATAFORMA_ATU: string (nullable = true)
 |-- B_RECARGA_COD_STATUS_PLATAFORMA: string (nullable = true)
 |-- B_RECARGA_IND_METODO_PAGAMENTO: string (nullable = true)
 |-- B_RECARGA_DW_PLANO_TARIFACAO: integer (nullable = true)
 |-- B_RECARGA_DW_TIPO_RECARGA: integer (nullable = true)
 |-- B_RECARGA_DW_

In [ ]:
df_principal_recarga.printSchema()

root
 |-- B_BUREAU_SAFRA: date (nullable = true)
 |-- B_BUREAU_Ano: integer (nullable = true)
 |-- B_BUREAU_Mes: integer (nullable = true)
 |-- B_BUREAU_IsInstallation: boolean (nullable = true)
 |-- B_BUREAU_ProductDescription: string (nullable = true)
 |-- B_BUREAU_ProductMigration: string (nullable = true)
 |-- B_BUREAU_Score01: float (nullable = true)
 |-- B_BUREAU_Score02: float (nullable = true)
 |-- B_BUREAU_FDP: integer (nullable = true)
 |-- B_BUREAU_NUM_CPF: string (nullable = true)
 |-- B_RECARGA_DW_NUM_NTC: string (nullable = true)
 |-- B_RECARGA_DW_NUM_CLIENTE: string (nullable = true)
 |-- B_RECARGA_DAT_INSERCAO_CREDITO: date (nullable = true)
 |-- B_RECARGA_HOR_INSERCAO_CREDITO: string (nullable = true)
 |-- B_RECARGA_COD_TECNOLOGIA_DW: string (nullable = true)
 |-- B_RECARGA_COD_CANAL_AQUISICAO: integer (nullable = true)
 |-- B_RECARGA_COD_TIPO_CREDITO: string (nullable = true)
 |-- B_RECARGA_COD_PROMOCAO: integer (nullable = true)
 |-- B_RECARGA_VAL_CREDITO_INSERIDO: f

# Carregando Base Recarga com Dados da Base Bureau

In [9]:
df_base_recarga01 = spark.read.parquet("/content/gdrive/MyDrive/Raw Hackathon PoD 2025/tabelao/tabela_recarga_data_de_recarga")
df_base_recarga01.createOrReplaceTempView("df_base_recarga")

# Analise Inicial

In [10]:
# Buscar cpfs repetidos na base e a quantidade de vezes que aparecem
df_base_recarga01.createOrReplaceTempView("base_for_multiples")
df_cpfs = spark.sql("""
SELECT
    B_BUREAU_NUM_CPF,
    COUNT(*) AS qtd_ocorrencias
FROM base_for_multiples
WHERE B_BUREAU_NUM_CPF IS NOT NULL
GROUP BY B_BUREAU_NUM_CPF
HAVING COUNT(*) > 1
ORDER BY qtd_ocorrencias DESC
""")

In [ ]:
df_base_recarga01.show(20, truncate=False)

+----------------+---------------+
|B_BUREAU_NUM_CPF|qtd_ocorrencias|
+----------------+---------------+
|UUWUWZXX879     |1183           |
|XYT9U8X778W     |717            |
|ZTWN798WXXX     |704            |
|XWW7T8XNUZT     |700            |
|7ZTUY779NX7     |582            |
|Z7N8NN9TTWZ     |482            |
|X87T78WW77Z     |477            |
|ZZ88UZXTY97     |470            |
|Z78Z87XUYXN     |469            |
|ZW797ZWZYT8     |443            |
|Z78YXX78YX7     |414            |
|UZT7XYY88TY     |406            |
|WUNYUYNT8WX     |402            |
|WXXWUX988WW     |398            |
|XUU8YWXN7X7     |394            |
|XWZU7YZ77YZ     |394            |
|ZZ9U8ZT7YTU     |386            |
|798T7UWUYXT     |383            |
|ZW7NNNYNXXN     |378            |
|XYZT9WWZ777     |378            |
+----------------+---------------+
only showing top 20 rows


In [11]:
df_base_recarga01.createOrReplaceTempView("base_for_multiples")

df_cpf_safra_qtd = spark.sql("""
SELECT
    B_BUREAU_NUM_CPF,
    B_BUREAU_SAFRA,
    COUNT(*) AS qtd_repeticoes_na_safra
FROM base_for_multiples
WHERE B_BUREAU_NUM_CPF IS NOT NULL
GROUP BY
    B_BUREAU_NUM_CPF,
    B_BUREAU_SAFRA
HAVING COUNT(*) > 1
ORDER BY
    qtd_repeticoes_na_safra DESC,
    B_BUREAU_NUM_CPF,
    B_BUREAU_SAFRA
""")




In [ ]:
df_cpf_safra_qtd.show(50, truncate=False)

+----------------+--------------+-----------------------+
|B_BUREAU_NUM_CPF|B_BUREAU_SAFRA|qtd_repeticoes_na_safra|
+----------------+--------------+-----------------------+
|ZTWN798WXXX     |2024-11-01    |704                    |
|ZW797ZWZYT8     |2025-03-01    |443                    |
|Z78YXX78YX7     |2024-12-01    |414                    |
|UUWUWZXX879     |2024-11-01    |379                    |
|ZW7NNNYNXXN     |2024-10-01    |378                    |
|UUWUWZXX879     |2025-01-01    |373                    |
|XYT9U8X778W     |2024-10-01    |372                    |
|ZYNT7N8WZNX     |2024-12-01    |362                    |
|XYT9U8X778W     |2025-01-01    |345                    |
|ZYN7Y8T7UUZ     |2024-12-01    |339                    |
|XY9U89977TT     |2025-01-01    |333                    |
|XXXNTTZ97Z9     |2024-10-01    |328                    |
|XUX8W8TN7ZX     |2025-03-01    |326                    |
|ZX9YWXZT779     |2024-12-01    |314                    |
|UUWUWZXX879  

In [12]:
# Selecionar o cpf que teve mais repetições para veririfcar como se comporta na base ou se teve valores duplicados

cpf_alvo = "XWYUTYWY7XW"

df_base_recarga01.createOrReplaceTempView("base_for_multiples")

df_cpf_detalhe = spark.sql(f"""
SELECT
    *
FROM base_for_multiples
WHERE B_BUREAU_NUM_CPF = '{cpf_alvo}'
ORDER BY B_BUREAU_SAFRA, B_RECARGA_DAT_INSERCAO_CREDITO
""")



In [ ]:
df_cpf_detalhe.show(20, truncate=False)


+------------+------------+-----------------------+---------------------------+-------------------------+----------------+----------------+------------+----------------+--------------------+------------------------+------------------------------+------------------------------+---------------------------+-----------------------------+--------------------------+----------------------+------------------------------+-------------------+------------------+----------------------------+-------------------------------+------------------------------+----------------------------+-------------------------+--------------------------+----------------------------+------------------------+--------------------------+------------------------------+------------------+-------------------+---------------+--------------+
|B_BUREAU_Ano|B_BUREAU_Mes|B_BUREAU_IsInstallation|B_BUREAU_ProductDescription|B_BUREAU_ProductMigration|B_BUREAU_Score01|B_BUREAU_Score02|B_BUREAU_FDP|B_BUREAU_NUM_CPF|B_RECARGA_DW_NUM_NT

In [13]:
# Verificar a cardinalidade, quantidade de nulos da tabela e percentuais

from pyspark.sql import functions as F

def gerar_metadado_spark(df):

    total_linhas = df.count()
    resultados = []

    for coluna, tipo in df.dtypes:

        col_ref = F.col(coluna)

        # trata null + string vazia
        cond_nulo = col_ref.isNull() | (F.trim(col_ref) == "")

        stats = df.agg(
            F.count(F.when(cond_nulo, 1)).alias("qtd_nulos"),
            F.countDistinct(col_ref).alias("cardinalidade")
        ).collect()[0]

        qtd_nulos = stats["qtd_nulos"]
        cardinalidade = stats["cardinalidade"]

        percentual_nulos = round((qtd_nulos / total_linhas) * 100, 2) if total_linhas > 0 else 0

        resultados.append(
            (
                coluna,
                tipo,
                qtd_nulos,
                percentual_nulos,
                cardinalidade
            )
        )

    return spark.createDataFrame(
        resultados,
        schema=[
            "nome_variavel",
            "tipo",
            "qtd_nulos",
            "percentual_nulos",
            "cardinalidade"
        ]
    ).orderBy("nome_variavel")



In [ ]:
metadado_df = gerar_metadado_spark(df_base_recarga01)
metadado_df.show(30,truncate=False)


+-------------------------------+-------+---------+----------------+-------------+
|nome_variavel                  |tipo   |qtd_nulos|percentual_nulos|cardinalidade|
+-------------------------------+-------+---------+----------------+-------------+
|B_BUREAU_Ano                   |int    |0        |0.0             |2            |
|B_BUREAU_FDP                   |int    |3429021  |28.61           |2            |
|B_BUREAU_IsInstallation        |boolean|0        |0.0             |2            |
|B_BUREAU_Mes                   |int    |0        |0.0             |6            |
|B_BUREAU_NUM_CPF               |string |0        |0.0             |3590459      |
|B_BUREAU_ProductDescription    |string |0        |0.0             |1            |
|B_BUREAU_ProductMigration      |string |3429021  |28.61           |3            |
|B_BUREAU_SAFRA                 |date   |0        |0.0             |6            |
|B_BUREAU_Score01               |float  |133467   |1.11            |308          |
|B_B

In [14]:
# contar CPF x FDP para ver se os nulos são dos mesmos CPFs
df_base_recarga01.createOrReplaceTempView("CPF_X_FDP")

df_cpf_fdp =spark.sql(f"""

SELECT
    B_BUREAU_NUM_CPF,
    COUNT(*) AS total_linhas,
    SUM(CASE WHEN B_BUREAU_FDP IS NULL THEN 1 ELSE 0 END) AS qtd_fdp_nulo
FROM CPF_X_FDP
GROUP BY B_BUREAU_NUM_CPF
HAVING qtd_fdp_nulo > 0

""")


In [ ]:
df_cpf_fdp.show(30, truncate=False)

+----------------+------------+------------+
|B_BUREAU_NUM_CPF|total_linhas|qtd_fdp_nulo|
+----------------+------------+------------+
|777YTW87T87     |1           |1           |
|7NX7879Z9XZ     |1           |1           |
|7YTZUXY9WZZ     |26          |26          |
|7ZN9TWTUXUZ     |13          |13          |
|8N79WYNYT99     |1           |1           |
|8UYNUTNZ9ZZ     |2           |2           |
|8WT7NUY8XZZ     |5           |5           |
|8X7Z8Y89NXT     |1           |1           |
|8Y978W87T7Y     |5           |5           |
|8YY97Z88Y7Y     |11          |11          |
|9979YUY7WN8     |1           |1           |
|9XZXX9ZNXZU     |3           |3           |
|N8N99N7ZYU9     |10          |10          |
|NNXU7XZ7XZZ     |3           |3           |
|NXYW8WNTWNX     |1           |1           |
|NZX7NWTZX87     |1           |1           |
|T889N8TZ8Y7     |1           |1           |
|T99NZU9W9YZ     |2           |2           |
|TUZ7ZXZ887U     |1           |1           |
|TZUUXTT87

In [15]:
# colocar os valores nulos de fdp como não migrou
df_base_recarga01.createOrReplaceTempView("base_for_multiples")

df_cpf_comportamento = spark.sql("""

SELECT
    B_BUREAU_NUM_CPF,

    COUNT(*) AS QTD_REGISTROS_TOTAL,

    -- Houve instalação e migração (FDP = 1)
    SUM(CASE WHEN B_BUREAU_FDP = 1 THEN 1 ELSE 0 END)
        AS QTD_INSTALOU_E_MIGROU,

    -- Houve instalação, mas não migrou (FDP = 0)
    SUM(CASE WHEN B_BUREAU_FDP = 0 THEN 1 ELSE 0 END)
        AS QTD_INSTALOU_E_NAO_MIGROU,

    -- Não houve instalação (FDP nulo)
    SUM(CASE WHEN B_BUREAU_FDP IS NULL THEN 1 ELSE 0 END)
        AS QTD_SEM_INSTALACAO,

    -- Flag: cliente teve FDP (migração) em algum momento
    CASE
        WHEN SUM(CASE WHEN B_BUREAU_FDP = 1 THEN 1 ELSE 0 END) > 0
            THEN 1
        ELSE 0
    END AS FLAG_TEVE_FDP

FROM base_for_multiples
GROUP BY B_BUREAU_NUM_CPF
HAVING COUNT(*) > 1
ORDER BY QTD_REGISTROS_TOTAL DESC
""")


In [ ]:

df_cpf_comportamento.show(20, truncate=False)

+----------------+-------------------+---------------------+-------------------------+------------------+-------------+
|B_BUREAU_NUM_CPF|QTD_REGISTROS_TOTAL|QTD_INSTALOU_E_MIGROU|QTD_INSTALOU_E_NAO_MIGROU|QTD_SEM_INSTALACAO|FLAG_TEVE_FDP|
+----------------+-------------------+---------------------+-------------------------+------------------+-------------+
|UUWUWZXX879     |1183               |0                    |810                      |373               |0            |
|XYT9U8X778W     |717                |0                    |717                      |0                 |0            |
|ZTWN798WXXX     |704                |0                    |0                        |704               |0            |
|XWW7T8XNUZT     |700                |0                    |0                        |700               |0            |
|7ZTUY779NX7     |582                |410                  |0                        |172               |1            |
|Z7N8NN9TTWZ     |482                |0 

# Analise de cpfs, safras, e distribuição de recarga.





In [16]:
# Quantidade de recargas por CPF e por mês (base mensal)
df_base_recarga01.createOrReplaceTempView("base_mensal_raw")

df_quantidade_de_recargas = spark.sql("""
WITH base_mensal_agg AS (
    SELECT
        B_BUREAU_NUM_CPF,

        TO_DATE(
            CONCAT(
                YEAR(B_RECARGA_DAT_INSERCAO_CREDITO), '-',
                LPAD(MONTH(B_RECARGA_DAT_INSERCAO_CREDITO), 2, '0'),
                '-01'
            )
        ) AS B_BUREAU_SAFRA,

        COUNT(*) AS QTD_RECARGAS_MES,

        CAST(
            SUM(COALESCE(B_RECARGA_VAL_REAL, 0))
            AS DECIMAL(18,2)
        ) AS VAL_TOTAL_MES

    FROM base_mensal_raw
    GROUP BY
        B_BUREAU_NUM_CPF,
        YEAR(B_RECARGA_DAT_INSERCAO_CREDITO),
        MONTH(B_RECARGA_DAT_INSERCAO_CREDITO)
)
SELECT
    B_BUREAU_NUM_CPF,
    B_BUREAU_SAFRA,
    YEAR(B_BUREAU_SAFRA) AS ANO_SAFRA,
    QTD_RECARGAS_MES,
    VAL_TOTAL_MES,

    SUM(QTD_RECARGAS_MES) OVER (
        PARTITION BY B_BUREAU_NUM_CPF, YEAR(B_BUREAU_SAFRA)
    ) AS QTD_RECARGAS_ANO,

    CAST(
        SUM(VAL_TOTAL_MES) OVER (
            PARTITION BY B_BUREAU_NUM_CPF, YEAR(B_BUREAU_SAFRA)
        )
        AS DECIMAL(18,2)
    ) AS VAL_TOTAL_ANO

FROM base_mensal_agg

""")

In [ ]:
df_quantidade_de_recargas.show(20, truncate=False)

+----------------+--------------+---------+----------------+-------------+----------------+-------------+
|B_BUREAU_NUM_CPF|B_BUREAU_SAFRA|ANO_SAFRA|QTD_RECARGAS_MES|VAL_TOTAL_MES|QTD_RECARGAS_ANO|VAL_TOTAL_ANO|
+----------------+--------------+---------+----------------+-------------+----------------+-------------+
|777778UZTN8     |2024-12-01    |2024     |7               |161031.00    |7               |161031.00    |
|777778Z89XT     |2024-12-01    |2024     |1               |1.00         |1               |1.00         |
|77777UZ9W87     |2025-03-01    |2025     |2               |80501.00     |2               |80501.00     |
|77777XZ77ZU     |2024-11-01    |2024     |3               |80500.39     |3               |80500.39     |
|77778TUTWU9     |2025-01-01    |2025     |3               |80501.00     |3               |80501.00     |
|77778TXZ987     |2024-11-01    |2024     |2               |80501.00     |2               |80501.00     |
|77778W8WTZZ     |2025-02-01    |2025     |3  

In [17]:
# quantos meses por CPF/ano

df_base_recarga01.createOrReplaceTempView("cpf_ano")

df_cpf_ano = spark.sql("""
SELECT
    B_BUREAU_NUM_CPF,
    YEAR(B_BUREAU_SAFRA) AS ANO,
    COUNT(DISTINCT B_BUREAU_SAFRA) AS QTD_MESES
FROM cpf_ano
WHERE B_BUREAU_SAFRA IS NOT NULL
GROUP BY
    B_BUREAU_NUM_CPF,
    YEAR(B_BUREAU_SAFRA)
""")


In [ ]:
df_cpf_ano.show(20, truncate=False)

+----------------+----+---------+
|B_BUREAU_NUM_CPF|ANO |QTD_MESES|
+----------------+----+---------+
|W7TZ8XN98Z8     |2024|1        |
|Z9NYUZTWUTN     |2024|1        |
|8Y7U79UN7YZ     |2024|1        |
|ZT89XW98788     |2024|1        |
|ZZNY88NXWUN     |2024|1        |
|ZNUW7YU8N9N     |2024|1        |
|YYUNUYN88ZX     |2024|1        |
|Z9UW77Y8TZZ     |2024|1        |
|ZU8WZXNTWT9     |2024|1        |
|ZT879UZ9U8X     |2024|1        |
|XTZ8UTNT777     |2024|1        |
|ZZ7UUZT8Y9T     |2024|1        |
|ZT8Z7WXTTW7     |2024|1        |
|7N7ZUYYZX78     |2024|1        |
|WU8ZNW9Y8W9     |2024|1        |
|NN897NY9YZZ     |2024|1        |
|UU9YZWXW8W7     |2024|1        |
|8YXU9XTXYZZ     |2024|1        |
|WUYT9ZZ8897     |2024|1        |
|ZY79W98TT7Z     |2024|1        |
+----------------+----+---------+
only showing top 20 rows


In [18]:
# distribuição das safras

df_base_recarga01.createOrReplaceTempView("safra")

df_safra = spark.sql("""
SELECT
    B_BUREAU_NUM_CPF,
    YEAR(B_BUREAU_SAFRA) AS ANO_SAFRA,
    COUNT(DISTINCT B_BUREAU_SAFRA) AS QTD_MESES_COM_DADOS,
    MIN(B_BUREAU_SAFRA) AS PRIMEIRA_SAFRA,
    MAX(B_BUREAU_SAFRA) AS ULTIMA_SAFRA
FROM safra
WHERE B_BUREAU_SAFRA IS NOT NULL
GROUP BY
    B_BUREAU_NUM_CPF,
    YEAR(B_BUREAU_SAFRA)
ORDER BY
    QTD_MESES_COM_DADOS ASC

""")


In [ ]:
df_safra.show(20, truncate=False)

+----------------+---------+-------------------+--------------+------------+
|B_BUREAU_NUM_CPF|ANO_SAFRA|QTD_MESES_COM_DADOS|PRIMEIRA_SAFRA|ULTIMA_SAFRA|
+----------------+---------+-------------------+--------------+------------+
|77779ZY9TU9     |2024     |1                  |2024-12-01    |2024-12-01  |
|7777NUXZUYZ     |2025     |1                  |2025-03-01    |2025-03-01  |
|7777UUN9YU9     |2025     |1                  |2025-01-01    |2025-01-01  |
|7777UZ78ZTW     |2024     |1                  |2024-10-01    |2024-10-01  |
|7777N7NWYN8     |2025     |1                  |2025-01-01    |2025-01-01  |
|7777NWXW7WU     |2024     |1                  |2024-12-01    |2024-12-01  |
|77777999T9X     |2024     |1                  |2024-12-01    |2024-12-01  |
|777778Z89XT     |2024     |1                  |2024-12-01    |2024-12-01  |
|7777NN9ZXZU     |2024     |1                  |2024-10-01    |2024-10-01  |
|7777NXUWZU9     |2025     |1                  |2025-03-01    |2025-03-01  |

In [19]:
# Distribuição de CPFs por quantidade de meses

df_base_recarga01.createOrReplaceTempView("cpf_safra")

df_cpf_safra = spark.sql("""

WITH meses_por_cpf AS (
    SELECT
        B_BUREAU_NUM_CPF,
        YEAR(B_BUREAU_SAFRA) AS ANO_SAFRA,
        COUNT(DISTINCT B_BUREAU_SAFRA) AS QTD_MESES
    FROM cpf_safra
    WHERE B_BUREAU_SAFRA IS NOT NULL
    GROUP BY
        B_BUREAU_NUM_CPF,
        YEAR(B_BUREAU_SAFRA)
)
SELECT
    ANO_SAFRA,
    QTD_MESES,
    COUNT(*) AS QTD_CPFS
FROM meses_por_cpf
GROUP BY ANO_SAFRA, QTD_MESES
ORDER BY ANO_SAFRA, QTD_MESES

""")

In [ ]:
df_cpf_safra.show(20, truncate=False)

+---------+---------+--------+
|ANO_SAFRA|QTD_MESES|QTD_CPFS|
+---------+---------+--------+
|2024     |1        |1803932 |
|2024     |2        |51876   |
|2024     |3        |1070    |
|2025     |1        |1781346 |
|2025     |2        |49837   |
|2025     |3        |1132    |
+---------+---------+--------+



2024

1 mês: 1.803.932 CPFs

2 meses: 51.876 CPFs

3 meses: 1.070 CPFs

2025

1 mês: 1.781.346 CPFs

2 meses: 49.837 CPFs

3 meses: 1.132 CPF

In [20]:
# Distribuição de CPFs por FPD, FPD = 1 → Inadimplente, FPD = 0 → Bom pagador, FPD IS NULL → Não elegível / não instalou

df_base_recarga01.createOrReplaceTempView("cpf_safra")

df_cpf_safra = spark.sql("""

WITH base_fpd AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_FDP
    FROM cpf_safra
),

cpf_controle AS (
    SELECT
        B_BUREAU_FDP,
        COUNT(*) AS QTD_REGISTROS,
        COUNT(DISTINCT B_BUREAU_NUM_CPF) AS QTD_CPFS
    FROM base_fpd
    GROUP BY B_BUREAU_FDP
),

cpf_repetidos AS (
    SELECT
        B_BUREAU_FDP,
        COUNT(*) AS QTD_CPFS_REPETIDOS
    FROM (
        SELECT
            B_BUREAU_FDP,
            B_BUREAU_NUM_CPF
        FROM base_fpd
        GROUP BY B_BUREAU_FDP, B_BUREAU_NUM_CPF
        HAVING COUNT(*) > 1
    )
    GROUP BY B_BUREAU_FDP
),

total_cpfs AS (
    SELECT
        COUNT(DISTINCT B_BUREAU_NUM_CPF) AS TOTAL_CPFS
    FROM base_fpd
)

SELECT
    c.B_BUREAU_FDP AS STATUS_FPD,

    c.QTD_REGISTROS,
    c.QTD_CPFS,

    COALESCE(r.QTD_CPFS_REPETIDOS, 0) AS QTD_CPFS_REPETIDOS,

    ROUND(
        (c.QTD_CPFS * 100.0) / t.TOTAL_CPFS,
        2
    ) AS PERC_CPFS

FROM cpf_controle c
LEFT JOIN cpf_repetidos r
    ON c.B_BUREAU_FDP = r.B_BUREAU_FDP
CROSS JOIN total_cpfs t

ORDER BY STATUS_FPD

""")

In [ ]:
df_cpf_safra.show(20, truncate=False)

+----------+-------------+--------+------------------+---------+
|STATUS_FPD|QTD_REGISTROS|QTD_CPFS|QTD_CPFS_REPETIDOS|PERC_CPFS|
+----------+-------------+--------+------------------+---------+
|NULL      |3429021      |1093008 |0                 |30.44    |
|0         |6484155      |2041028 |1482652           |56.85    |
|1         |2074157      |546809  |463492            |15.23    |
+----------+-------------+--------+------------------+---------+



Distribuição do FPD — Análise

 Visão Geral

STATUS_FPD	Interpretação	% CPFs

NULL	Não elegível (sem instalação)	30,44%

0	Bom pagador	56,85%

1	Inadimplente (FPD)	15,23%

# Criação de Books Recarga

In [22]:
# Safra com mais recargas por CPF
df_base_recarga01.createOrReplaceTempView("base_for_multiples")

df_safras_quantidade_recarga = spark.sql(f"""

WITH base_mensal AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        COUNT(*) AS QTD_RECARGAS_MES
    FROM base_for_multiples
    GROUP BY B_BUREAU_NUM_CPF, B_BUREAU_SAFRA
),
rank_safra AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY B_BUREAU_NUM_CPF
               ORDER BY QTD_RECARGAS_MES DESC
           ) AS rn
    FROM base_mensal
)
SELECT
    B_BUREAU_NUM_CPF,
    B_BUREAU_SAFRA AS SAFRA_MAIS_RECARGAS,
    QTD_RECARGAS_MES
FROM rank_safra
WHERE rn == 1

""")

In [ ]:
df_safras_quantidade_recarga.show(50, truncate=False)

+----------------+-------------------+----------------+
|B_BUREAU_NUM_CPF|SAFRA_MAIS_RECARGAS|QTD_RECARGAS_MES|
+----------------+-------------------+----------------+
|777778Z89XT     |2024-12-01         |1               |
|77777U9YN9X     |2025-03-01         |4               |
|77777WUUUZU     |2025-03-01         |2               |
|77777XZ77ZU     |2024-11-01         |3               |
|77777Z9W99X     |2024-10-01         |1               |
|777787T8N87     |2025-02-01         |3               |
|777789Y8TXT     |2025-03-01         |3               |
|77778NZZUNW     |2025-01-01         |2               |
|77778W8WTZZ     |2025-02-01         |3               |
|77778X7ZWTW     |2024-11-01         |6               |
|77778Y78XYZ     |2025-01-01         |1               |
|77778YWTT87     |2025-02-01         |2               |
|77778YY78ZZ     |2024-12-01         |3               |
|77778Z9ZUZN     |2024-11-01         |2               |
|777797YZ7WZ     |2024-11-01         |2         

In [21]:
# Criar BOOKS de 3, 6, 9 e 12 meses

df_base_recarga01.createOrReplaceTempView("book_mes")

df_book_mensal = spark.sql("""
WITH base_mensal AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        COUNT(*) AS QTD_RECARGAS_MES,
        CAST(SUM(COALESCE(B_RECARGA_VAL_REAL, 0)) AS DECIMAL(18,2)) AS VAL_TOTAL_MES
    FROM book_mes
    WHERE B_BUREAU_SAFRA IS NOT NULL
    GROUP BY
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA
)

SELECT
    B_BUREAU_NUM_CPF,
    B_BUREAU_SAFRA,
    QTD_RECARGAS_MES,
    VAL_TOTAL_MES,

    -- BOOK 3 meses
    SUM(QTD_RECARGAS_MES) OVER (
        PARTITION BY B_BUREAU_NUM_CPF
        ORDER BY B_BUREAU_SAFRA
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS BOOK_3M,

    -- BOOK 6 meses
    SUM(QTD_RECARGAS_MES) OVER (
        PARTITION BY B_BUREAU_NUM_CPF
        ORDER BY B_BUREAU_SAFRA
        ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
    ) AS BOOK_6M,

    -- BOOK 12 meses
    SUM(QTD_RECARGAS_MES) OVER (
        PARTITION BY B_BUREAU_NUM_CPF
        ORDER BY B_BUREAU_SAFRA
        ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
    ) AS BOOK_12M

FROM base_mensal
ORDER BY
    B_BUREAU_NUM_CPF,
    B_BUREAU_SAFRA
""")


In [ ]:
df_book_mensal.show(50, truncate=False)

+----------------+--------------+----------------+-------------+-------+-------+--------+
|B_BUREAU_NUM_CPF|B_BUREAU_SAFRA|QTD_RECARGAS_MES|VAL_TOTAL_MES|BOOK_3M|BOOK_6M|BOOK_12M|
+----------------+--------------+----------------+-------------+-------+-------+--------+
|777777UWTYZ     |2025-02-01    |3               |30.00        |3      |3      |3       |
|777777UWTYZ     |2025-03-01    |5               |80520.00     |8      |8      |8       |
|777778UZTN8     |2024-12-01    |7               |161031.00    |7      |7      |7       |
|777778Z89XT     |2024-12-01    |1               |1.00         |1      |1      |1       |
|77777999T9X     |2024-12-01    |4               |80529.00     |4      |4      |4       |
|77777T8XTYZ     |2025-03-01    |2               |45.00        |2      |2      |2       |
|77777U9YN9X     |2025-03-01    |4               |80500.00     |4      |4      |4       |
|77777UZ9W87     |2025-03-01    |2               |80501.00     |2      |2      |2       |
|77777WUUU

In [ ]:
#book de recarga por safra e cpf
df_base_recarga01.createOrReplaceTempView("safra_cpf")

df_dataset_final = spark.sql("""
WITH base_mensal AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        COUNT(*) AS QTD_RECARGAS_MES,
        SUM(B_RECARGA_VAL_REAL) AS VAL_TOTAL_MES
    FROM safra_cpf
    GROUP BY B_BUREAU_NUM_CPF, B_BUREAU_SAFRA
),

books AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        QTD_RECARGAS_MES,
        VAL_TOTAL_MES,

        -- BOOKS DE RECARGA
        SUM(QTD_RECARGAS_MES) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS BOOK_3M_RECARGAS,

        SUM(QTD_RECARGAS_MES) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS BOOK_6M_RECARGAS,

        SUM(QTD_RECARGAS_MES) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
        ) AS BOOK_9M_RECARGAS,

        SUM(QTD_RECARGAS_MES) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS BOOK_12M_RECARGAS
    FROM base_mensal
),

safra_rank AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        QTD_RECARGAS_MES,
        ROW_NUMBER() OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY QTD_RECARGAS_MES DESC
        ) AS rn
    FROM base_mensal
),

safra_max AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA AS SAFRA_MAIS_RECARGAS,
        QTD_RECARGAS_MES AS QTD_RECARGAS_SAFRA_MAX
    FROM safra_rank
    WHERE rn = 1
)

SELECT
    b.B_BUREAU_NUM_CPF,
    b.B_BUREAU_SAFRA,

    b.QTD_RECARGAS_MES,
    b.VAL_TOTAL_MES,

    b.BOOK_3M_RECARGAS,
    b.BOOK_6M_RECARGAS,
    b.BOOK_9M_RECARGAS,
    b.BOOK_12M_RECARGAS,

    s.SAFRA_MAIS_RECARGAS,
    s.QTD_RECARGAS_SAFRA_MAX

FROM books b
LEFT JOIN safra_max s
    ON b.B_BUREAU_NUM_CPF = s.B_BUREAU_NUM_CPF

ORDER BY
    b.B_BUREAU_NUM_CPF,
    b.B_BUREAU_SAFRA
""")


In [ ]:
df_dataset_final.show(20, truncate=False)

+----------------+--------------+----------------+----------------+----------------+----------------+----------------+-----------------+-------------------+----------------------+
|B_BUREAU_NUM_CPF|B_BUREAU_SAFRA|QTD_RECARGAS_MES|VAL_TOTAL_MES   |BOOK_3M_RECARGAS|BOOK_6M_RECARGAS|BOOK_9M_RECARGAS|BOOK_12M_RECARGAS|SAFRA_MAIS_RECARGAS|QTD_RECARGAS_SAFRA_MAX|
+----------------+--------------+----------------+----------------+----------------+----------------+----------------+-----------------+-------------------+----------------------+
|777777UWTYZ     |2025-02-01    |3               |30.0            |3               |3               |3               |3                |2025-03-01         |5                     |
|777777UWTYZ     |2025-03-01    |5               |80520.0         |8               |8               |8               |8                |2025-03-01         |5                     |
|777778UZTN8     |2024-12-01    |7               |161031.0        |7               |7               

In [ ]:
# tabela com book e fpd Target: FPD (First Payment Default) - 0 = Bom pagador, 1 = Inadimplente
df_base_recarga01.createOrReplaceGlobalTempView("base_for_multiples")

df_book_fpd = spark.sql("""

WITH base_mensal AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        COUNT(*) AS QTD_RECARGAS_MES,
        CAST(SUM(COALESCE(B_RECARGA_VAL_REAL, 0)) AS DECIMAL(18,2)) AS VAL_TOTAL_MES
    FROM base_for_multiples
    GROUP BY
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA
),

book_recargas AS (
    SELECT
        *,
        SUM(QTD_RECARGAS_MES) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS BOOK_3M,

        SUM(QTD_RECARGAS_MES) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS BOOK_6M,

        SUM(QTD_RECARGAS_MES) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS BOOK_12M
    FROM base_mensal
),

fpd_safra AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        MAX(B_BUREAU_FDP) AS TARGET_FPD
    FROM base_for_multiples
    GROUP BY
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA
),

controle_temporal AS (
    SELECT
        B_BUREAU_NUM_CPF,
        YEAR(B_BUREAU_SAFRA) AS ANO_SAFRA,
        COUNT(DISTINCT B_BUREAU_SAFRA) AS QTD_MESES_ANO
    FROM base_for_multiples
    WHERE B_BUREAU_SAFRA IS NOT NULL
    GROUP BY
        B_BUREAU_NUM_CPF,
        YEAR(B_BUREAU_SAFRA)
)

SELECT
    b.B_BUREAU_NUM_CPF,
    b.B_BUREAU_SAFRA,
    YEAR(b.B_BUREAU_SAFRA) AS ANO_SAFRA,

    -- Recargas
    b.QTD_RECARGAS_MES,
    b.VAL_TOTAL_MES,
    b.BOOK_3M,
    b.BOOK_6M,
    b.BOOK_12M,

    -- Target
    f.TARGET_FPD,

    -- Controle de histórico
    c.QTD_MESES_ANO,
    CASE WHEN c.QTD_MESES_ANO >= 3 THEN 1 ELSE 0 END AS TEM_HIST_3M,
    CASE WHEN c.QTD_MESES_ANO >= 6 THEN 1 ELSE 0 END AS TEM_HIST_6M,
    CASE WHEN c.QTD_MESES_ANO >= 12 THEN 1 ELSE 0 END AS TEM_HIST_12M

FROM book_recargas b
LEFT JOIN fpd_safra f
    ON b.B_BUREAU_NUM_CPF = f.B_BUREAU_NUM_CPF
   AND b.B_BUREAU_SAFRA   = f.B_BUREAU_SAFRA
LEFT JOIN controle_temporal c
    ON b.B_BUREAU_NUM_CPF = c.B_BUREAU_NUM_CPF
   AND YEAR(b.B_BUREAU_SAFRA) = c.ANO_SAFRA
ORDER BY
    b.B_BUREAU_NUM_CPF,
    b.B_BUREAU_SAFRA

    """)


In [ ]:
df_book_fpd.show(20, truncate=False)

+----------------+--------------+---------+----------------+-------------+-------+-------+--------+----------+-------------+-----------+-----------+------------+
|B_BUREAU_NUM_CPF|B_BUREAU_SAFRA|ANO_SAFRA|QTD_RECARGAS_MES|VAL_TOTAL_MES|BOOK_3M|BOOK_6M|BOOK_12M|TARGET_FPD|QTD_MESES_ANO|TEM_HIST_3M|TEM_HIST_6M|TEM_HIST_12M|
+----------------+--------------+---------+----------------+-------------+-------+-------+--------+----------+-------------+-----------+-----------+------------+
|777777UWTYZ     |2025-02-01    |2025     |3               |30.00        |3      |3      |3       |NULL      |2            |0          |0          |0           |
|777777UWTYZ     |2025-03-01    |2025     |5               |80520.00     |8      |8      |8       |0         |2            |0          |0          |0           |
|777778UZTN8     |2024-12-01    |2024     |7               |161031.00    |7      |7      |7       |0         |1            |0          |0          |0           |
|777778Z89XT     |2024-12-01

In [11]:
# book CPF + SAFRA + FDP  book versao 1
df_base_recarga01.createOrReplaceTempView("cpf_safra_fpd")

df_dataset_fdp = spark.sql("""
WITH base_mensal AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        B_BUREAU_FDP,

        COUNT(*) AS QTD_RECARGAS_MES,
        SUM(B_RECARGA_VAL_REAL) AS VAL_TOTAL_MES
    FROM cpf_safra_fpd
    GROUP BY
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        B_BUREAU_FDP
),

books AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        B_BUREAU_FDP,

        QTD_RECARGAS_MES,
        VAL_TOTAL_MES,

        -- BOOKS CONDICIONADOS AO FDP
        SUM(QTD_RECARGAS_MES) OVER (
            PARTITION BY B_BUREAU_NUM_CPF, B_BUREAU_FDP
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS BOOK_3M_RECARGAS,

        SUM(QTD_RECARGAS_MES) OVER (
            PARTITION BY B_BUREAU_NUM_CPF, B_BUREAU_FDP
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS BOOK_6M_RECARGAS,

        SUM(QTD_RECARGAS_MES) OVER (
            PARTITION BY B_BUREAU_NUM_CPF, B_BUREAU_FDP
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS BOOK_12M_RECARGAS
    FROM base_mensal
),

fdp_rank AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_FDP,
        SUM(QTD_RECARGAS_MES) AS QTD_TOTAL_RECARGAS,
        ROW_NUMBER() OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY SUM(QTD_RECARGAS_MES) DESC
        ) AS rn
    FROM base_mensal
    GROUP BY
        B_BUREAU_NUM_CPF,
        B_BUREAU_FDP
),

fdp_max AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_FDP AS FDP_MAIS_RECARGAS,
        QTD_TOTAL_RECARGAS
    FROM fdp_rank
    WHERE rn = 1
)

SELECT
    b.B_BUREAU_NUM_CPF,
    b.B_BUREAU_SAFRA,
    b.B_BUREAU_FDP,

    b.QTD_RECARGAS_MES,
    b.VAL_TOTAL_MES,

    b.BOOK_3M_RECARGAS,
    b.BOOK_6M_RECARGAS,
    b.BOOK_12M_RECARGAS,

    f.FDP_MAIS_RECARGAS,
    f.QTD_TOTAL_RECARGAS

FROM books b
LEFT JOIN fdp_max f
    ON b.B_BUREAU_NUM_CPF = f.B_BUREAU_NUM_CPF

ORDER BY
    b.B_BUREAU_NUM_CPF,
    b.B_BUREAU_SAFRA,
    b.B_BUREAU_FDP
""")


In [ ]:
df_dataset_fdp.show(20, truncate=False)

+----------------+--------------+------------+----------------+----------------+----------------+----------------+-----------------+-----------------+------------------+
|B_BUREAU_NUM_CPF|B_BUREAU_SAFRA|B_BUREAU_FDP|QTD_RECARGAS_MES|VAL_TOTAL_MES   |BOOK_3M_RECARGAS|BOOK_6M_RECARGAS|BOOK_12M_RECARGAS|FDP_MAIS_RECARGAS|QTD_TOTAL_RECARGAS|
+----------------+--------------+------------+----------------+----------------+----------------+----------------+-----------------+-----------------+------------------+
|777777UWTYZ     |2025-02-01    |NULL        |3               |30.0            |3               |3               |3                |0                |5                 |
|777777UWTYZ     |2025-03-01    |0           |5               |80520.0         |5               |5               |5                |0                |5                 |
|777778UZTN8     |2024-12-01    |0           |7               |161031.0        |7               |7               |7                |0                |

In [12]:
# book veersao 1 melhorado com ajustes
df_base_recarga01.createOrReplaceTempView("base_for_multiples")

df_dataset_fdp = spark.sql("""
WITH base_mensal AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        B_BUREAU_FDP,

        COUNT(*) AS QTD_RECARGAS_MES,

        CAST(
            SUM(COALESCE(B_RECARGA_VAL_REAL, 0))
            AS DECIMAL(18,2)
        ) AS VAL_TOTAL_MES,

        CAST(
            SUM(COALESCE(B_RECARGA_VALOR_SOS, 0))
            AS DECIMAL(18,2)
        ) AS VAL_TOTAL_SOS_MES

    FROM base_for_multiples
    GROUP BY
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        B_BUREAU_FDP
),

books AS (
    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        B_BUREAU_FDP,

        QTD_RECARGAS_MES,
        VAL_TOTAL_MES,
        VAL_TOTAL_SOS_MES,

        -- BOOKS DE QUANTIDADE
        SUM(QTD_RECARGAS_MES) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS BOOK_3M_QTD,

        SUM(QTD_RECARGAS_MES) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS BOOK_6M_QTD,

        SUM(QTD_RECARGAS_MES) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS BOOK_12M_QTD,

        -- BOOKS DE VALOR
        CAST(
            SUM(VAL_TOTAL_MES) OVER (
                PARTITION BY B_BUREAU_NUM_CPF
                ORDER BY B_BUREAU_SAFRA
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) AS DECIMAL(18,2)
        ) AS BOOK_3M_VAL,

        CAST(
            SUM(VAL_TOTAL_MES) OVER (
                PARTITION BY B_BUREAU_NUM_CPF
                ORDER BY B_BUREAU_SAFRA
                ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
            ) AS DECIMAL(18,2)
        ) AS BOOK_6M_VAL,

        CAST(
            SUM(VAL_TOTAL_MES) OVER (
                PARTITION BY B_BUREAU_NUM_CPF
                ORDER BY B_BUREAU_SAFRA
                ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
            ) AS DECIMAL(18,2)
        ) AS BOOK_12M_VAL,

        -- BOOKS DE SOS
        CAST(
            SUM(VAL_TOTAL_SOS_MES) OVER (
                PARTITION BY B_BUREAU_NUM_CPF
                ORDER BY B_BUREAU_SAFRA
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) AS DECIMAL(18,2)
        ) AS BOOK_3M_SOS,

        CAST(
            SUM(VAL_TOTAL_SOS_MES) OVER (
                PARTITION BY B_BUREAU_NUM_CPF
                ORDER BY B_BUREAU_SAFRA
                ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
            ) AS DECIMAL(18,2)
        ) AS BOOK_6M_SOS,

        CAST(
            SUM(VAL_TOTAL_SOS_MES) OVER (
                PARTITION BY B_BUREAU_NUM_CPF
                ORDER BY B_BUREAU_SAFRA
                ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
            ) AS DECIMAL(18,2)
        ) AS BOOK_12M_SOS

    FROM base_mensal
)

SELECT
    *
FROM books
ORDER BY
    B_BUREAU_NUM_CPF,
    B_BUREAU_SAFRA,
    B_BUREAU_FDP

""")


In [ ]:
df_dataset_fdp.show(20, truncate=False)

+----------------+--------------+------------+----------------+-------------+-----------------+-----------+-----------+------------+-----------+-----------+------------+-----------+-----------+------------+
|B_BUREAU_NUM_CPF|B_BUREAU_SAFRA|B_BUREAU_FDP|QTD_RECARGAS_MES|VAL_TOTAL_MES|VAL_TOTAL_SOS_MES|BOOK_3M_QTD|BOOK_6M_QTD|BOOK_12M_QTD|BOOK_3M_VAL|BOOK_6M_VAL|BOOK_12M_VAL|BOOK_3M_SOS|BOOK_6M_SOS|BOOK_12M_SOS|
+----------------+--------------+------------+----------------+-------------+-----------------+-----------+-----------+------------+-----------+-----------+------------+-----------+-----------+------------+
|777777UWTYZ     |2025-02-01    |NULL        |3               |30.00        |0.00             |3          |3          |3           |30.00      |30.00      |30.00       |0.00       |0.00       |0.00        |
|777777UWTYZ     |2025-03-01    |0           |5               |80520.00     |0.00             |8          |8          |8           |80550.00   |80550.00   |80550.00    |0.0

In [19]:
#BOOK COM VERSAO MAIS ROBUSTA
df_base_recarga01.createOrReplaceTempView("base_for_multiples")

df_dataset_fdp = spark.sql("""

WITH base_mensal AS (

    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        B_BUREAU_FDP,

        -- MÊS DE REFERÊNCIA
        B_BUREAU_SAFRA AS MES_REFERENCIA,

        -- RECARGA
        COUNT(*) AS QTD_RECARGAS_MES,

        CAST(
            SUM(COALESCE(B_RECARGA_VAL_REAL, 0))
            AS DECIMAL(18,2)
        ) AS VAL_TOTAL_RECARGA_MES,

        CASE
            WHEN COUNT(*) > 0 THEN 1
            ELSE 0
        END AS FLAG_MES_RECARGA,

        -- SOS
        CAST(
            SUM(COALESCE(B_RECARGA_VALOR_SOS, 0))
            AS DECIMAL(18,2)
        ) AS VAL_TOTAL_SOS_MES,

        CASE
            WHEN SUM(COALESCE(B_RECARGA_VALOR_SOS, 0)) > 0 THEN 1
            ELSE 0
        END AS FLAG_MES_SOS,

        CASE
            WHEN SUM(COALESCE(B_RECARGA_VALOR_SOS, 0)) > 0
            THEN B_BUREAU_SAFRA
            ELSE NULL
        END AS MES_SOS

    FROM base_for_multiples
    GROUP BY
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        B_BUREAU_FDP
),

books AS (

    SELECT
        B_BUREAU_NUM_CPF,
        B_BUREAU_SAFRA,
        B_BUREAU_FDP,
        MES_REFERENCIA,

        -- BASE MENSAL
        QTD_RECARGAS_MES,
        VAL_TOTAL_RECARGA_MES,
        FLAG_MES_RECARGA,

        VAL_TOTAL_SOS_MES,
        FLAG_MES_SOS,
        MES_SOS,

        -- =========================
        -- RECARGA | MESES
        -- =========================
        SUM(FLAG_MES_RECARGA) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS QTD_RECARGA_3M,

        SUM(FLAG_MES_RECARGA) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS QTD_RECARGA_6M,

        SUM(FLAG_MES_RECARGA) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
        ) AS QTD_RECARGA_9M,

        SUM(FLAG_MES_RECARGA) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS QTD_RECARGA_12M,

        -- =========================
        -- RECARGA | VALOR
        -- =========================
        CAST(
            SUM(VAL_TOTAL_RECARGA_MES) OVER (
                PARTITION BY B_BUREAU_NUM_CPF
                ORDER BY B_BUREAU_SAFRA
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) AS DECIMAL(18,2)
        ) AS VAL_RECARGA_3M,

        CAST(
            SUM(VAL_TOTAL_RECARGA_MES) OVER (
                PARTITION BY B_BUREAU_NUM_CPF
                ORDER BY B_BUREAU_SAFRA
                ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
            ) AS DECIMAL(18,2)
        ) AS VAL_RECARGA_6M,

        CAST(
            SUM(VAL_TOTAL_RECARGA_MES) OVER (
                PARTITION BY B_BUREAU_NUM_CPF
                ORDER BY B_BUREAU_SAFRA
                ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
            ) AS DECIMAL(18,2)
        ) AS VAL_RECARGA_9M,

        CAST(
            SUM(VAL_TOTAL_RECARGA_MES) OVER (
                PARTITION BY B_BUREAU_NUM_CPF
                ORDER BY B_BUREAU_SAFRA
                ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
            ) AS DECIMAL(18,2)
        ) AS VAL_RECARGA_12M,

        -- =========================
        -- SOS | MESES
        -- =========================
        SUM(FLAG_MES_SOS) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS QTD_SOS_3M,

        SUM(FLAG_MES_SOS) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS QTD_SOS_6M,

        SUM(FLAG_MES_SOS) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
        ) AS QTD_SOS_9M,

        SUM(FLAG_MES_SOS) OVER (
            PARTITION BY B_BUREAU_NUM_CPF
            ORDER BY B_BUREAU_SAFRA
            ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
        ) AS QTD_SOS_12M,

        -- =========================
        -- SOS | VALOR
        -- =========================
        CAST(
            SUM(VAL_TOTAL_SOS_MES) OVER (
                PARTITION BY B_BUREAU_NUM_CPF
                ORDER BY B_BUREAU_SAFRA
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) AS DECIMAL(18,2)
        ) AS VALOR_SOS_3M,

        CAST(
            SUM(VAL_TOTAL_SOS_MES) OVER (
                PARTITION BY B_BUREAU_NUM_CPF
                ORDER BY B_BUREAU_SAFRA
                ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
            ) AS DECIMAL(18,2)
        ) AS VALOR_SOS_6M,

        CAST(
            SUM(VAL_TOTAL_SOS_MES) OVER (
                PARTITION BY B_BUREAU_NUM_CPF
                ORDER BY B_BUREAU_SAFRA
                ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
            ) AS DECIMAL(18,2)
        ) AS VALOR_SOS_9M,

        CAST(
            SUM(VAL_TOTAL_SOS_MES) OVER (
                PARTITION BY B_BUREAU_NUM_CPF
                ORDER BY B_BUREAU_SAFRA
                ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
            ) AS DECIMAL(18,2)
        ) AS VALOR_SOS_12M

    FROM base_mensal
)

SELECT *
FROM books
ORDER BY
    B_BUREAU_NUM_CPF,
    B_BUREAU_SAFRA,
    B_BUREAU_FDP


""")

In [20]:
df_dataset_fdp.show(20, truncate=False)

+----------------+--------------+------------+--------------+----------------+---------------------+----------------+-----------------+------------+-------+--------------+--------------+--------------+---------------+--------------+--------------+--------------+---------------+----------+----------+----------+-----------+------------+------------+------------+-------------+
|B_BUREAU_NUM_CPF|B_BUREAU_SAFRA|B_BUREAU_FDP|MES_REFERENCIA|QTD_RECARGAS_MES|VAL_TOTAL_RECARGA_MES|FLAG_MES_RECARGA|VAL_TOTAL_SOS_MES|FLAG_MES_SOS|MES_SOS|QTD_RECARGA_3M|QTD_RECARGA_6M|QTD_RECARGA_9M|QTD_RECARGA_12M|VAL_RECARGA_3M|VAL_RECARGA_6M|VAL_RECARGA_9M|VAL_RECARGA_12M|QTD_SOS_3M|QTD_SOS_6M|QTD_SOS_9M|QTD_SOS_12M|VALOR_SOS_3M|VALOR_SOS_6M|VALOR_SOS_9M|VALOR_SOS_12M|
+----------------+--------------+------------+--------------+----------------+---------------------+----------------+-----------------+------------+-------+--------------+--------------+--------------+---------------+--------------+--------------

In [23]:
# unir a tabela book pagamento na tabela recarga

df_base_recarga01.createOrReplaceTempView("recarga")
df_dataset_fdp.createOrReplaceTempView("book")

df_recarga_com_book = spark.sql("""

SELECT
    r.*,

    -- =========================
    -- BOOK RECARGA
    -- =========================
    b.QTD_RECARGA_3M,
    b.QTD_RECARGA_6M,
    b.QTD_RECARGA_9M,
    b.QTD_RECARGA_12M,

    b.VAL_RECARGA_3M,
    b.VAL_RECARGA_6M,
    b.VAL_RECARGA_9M,
    b.VAL_RECARGA_12M,

    -- =========================
    -- BOOK SOS
    -- =========================
    b.QTD_SOS_3M,
    b.QTD_SOS_6M,
    b.QTD_SOS_9M,
    b.QTD_SOS_12M,

    b.VALOR_SOS_3M,
    b.VALOR_SOS_6M,
    b.VALOR_SOS_9M,
    b.VALOR_SOS_12M

FROM recarga r
LEFT JOIN book b
    ON r.B_BUREAU_NUM_CPF = b.B_BUREAU_NUM_CPF
   AND r.B_BUREAU_SAFRA   = b.B_BUREAU_SAFRA
   AND r.B_BUREAU_FDP     = b.B_BUREAU_FDP
""")


In [25]:
df_recarga_com_book.show(20, truncate=False)

+------------+------------+-----------------------+---------------------------+-------------------------+----------------+----------------+------------+----------------+--------------------+------------------------+------------------------------+------------------------------+---------------------------+-----------------------------+--------------------------+----------------------+------------------------------+-------------------+------------------+----------------------------+-------------------------------+------------------------------+----------------------------+-------------------------+--------------------------+----------------------------+------------------------+--------------------------+------------------------------+------------------+-------------------+---------------+--------------+--------------+--------------+--------------+---------------+--------------+--------------+--------------+---------------+----------+----------+----------+-----------+------------+---------